##Load + Split

In [3]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.arushna_tremor_detector import preprocess_and_window

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

csv_path = r"C:\Users\arush\OneDrive\Desktop\ntut feb 2025\dataset\Dataset.csv"
X, y = preprocess_and_window(csv_path)
print("Loaded:", X.shape, y.shape)

# Stratified split (since you're not time-splitting now)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)
print("Split:", X_train.shape, X_val.shape, X_test.shape)
print("Counts:", np.bincount(y_train), np.bincount(y_val), np.bincount(y_test))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def make_loader(X_np, y_np, batch_size=64, shuffle=False):
    X_t = torch.tensor(X_np, dtype=torch.float32)
    y_t = torch.tensor(y_np, dtype=torch.long)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, batch_size=64, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   batch_size=64, shuffle=False)
test_loader  = make_loader(X_test,  y_test,  batch_size=64, shuffle=False)

Loaded: (956, 300, 3) (956,)
Split: (764, 300, 3) (96, 300, 3) (96, 300, 3)
Counts: [356 408] [44 52] [45 51]
Device: cpu


## CNN w/ embedding layer

In [4]:
class CNNFeatureNet(nn.Module):
    def __init__(self, in_ch=3, emb_dim=64):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)  # (B,64,25) -> (B,64,1)
        self.embedding = nn.Linear(64, emb_dim)
        self.classifier = nn.Linear(emb_dim, 2)

    def forward(self, x):
        # x: (B, T, C) -> (B, C, T)
        x = x.permute(0, 2, 1)
        h = self.feature_extractor(x)
        h = self.pool(h).squeeze(-1)         # (B,64)
        z = self.embedding(h)                # (B,emb_dim)
        z = torch.relu(z)
        logits = self.classifier(z)          # (B,2)
        return logits, z

model = CNNFeatureNet(in_ch=3, emb_dim=64).to(device)

counts = np.bincount(y_train)
weights = torch.tensor([1/counts[0], 1/counts[1]], dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

##Training Loop

In [5]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    all_preds, all_y = [], []
    total_loss = 0.0

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            if train:
                optimizer.zero_grad()

            logits, _ = model(xb)
            loss = criterion(logits, yb)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            preds = torch.argmax(logits, dim=1)
            all_preds.append(preds.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_y = np.concatenate(all_y)
    avg_loss = total_loss / len(all_y)
    acc = accuracy_score(all_y, all_preds)
    return avg_loss, acc

best_val_acc = -1
best_state = None

for epoch in range(1, 21):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)

    print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} acc {train_acc:.3f} | val loss {val_loss:.4f} acc {val_acc:.3f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print("Best val acc:", best_val_acc)
model.load_state_dict(best_state)
model.to(device)

Epoch 01 | train loss 6.3270 acc 0.518 | val loss 1.8188 acc 0.521
Epoch 02 | train loss 1.3143 acc 0.645 | val loss 0.8356 acc 0.729
Epoch 03 | train loss 0.8666 acc 0.704 | val loss 0.6645 acc 0.802
Epoch 04 | train loss 0.6118 acc 0.734 | val loss 0.5302 acc 0.792
Epoch 05 | train loss 0.4741 acc 0.787 | val loss 0.5643 acc 0.740
Epoch 06 | train loss 0.3900 acc 0.829 | val loss 0.4557 acc 0.771
Epoch 07 | train loss 0.3567 acc 0.840 | val loss 0.5443 acc 0.802
Epoch 08 | train loss 0.4582 acc 0.810 | val loss 0.3418 acc 0.854
Epoch 09 | train loss 0.3634 acc 0.852 | val loss 0.3426 acc 0.875
Epoch 10 | train loss 0.3015 acc 0.877 | val loss 0.5042 acc 0.812
Epoch 11 | train loss 0.3052 acc 0.877 | val loss 0.3227 acc 0.854
Epoch 12 | train loss 0.2801 acc 0.882 | val loss 0.3196 acc 0.844
Epoch 13 | train loss 0.3196 acc 0.863 | val loss 0.3078 acc 0.854
Epoch 14 | train loss 0.2549 acc 0.899 | val loss 0.3258 acc 0.896
Epoch 15 | train loss 0.2443 acc 0.901 | val loss 0.2670 acc 0

CNNFeatureNet(
  (feature_extractor): Sequential(
    (0): Conv1d(3, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (4): ReLU()
    (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (pool): AdaptiveAvgPool1d(output_size=1)
  (embedding): Linear(in_features=64, out_features=64, bias=True)
  (classifier): Linear(in_features=64, out_features=2, bias=True)
)

##Eval Test

In [6]:
import numpy as np
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

test_loss, test_acc = run_epoch(test_loader, train=False)
print("Test acc (argmax):", test_acc)

model.eval()
all_probs1, all_y = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits, _ = model(xb)

        # probability of class 1 (shaking)
        probs1 = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_probs1.append(probs1)
        all_y.append(yb.numpy())

all_probs1 = np.concatenate(all_probs1)
all_y = np.concatenate(all_y)

thr = 0.4
all_preds = (all_probs1 >= thr).astype(int)

print(f"Test acc (threshold={thr}):", accuracy_score(all_y, all_preds))
print(classification_report(all_y, all_preds))
print("Confusion matrix:\n", confusion_matrix(all_y, all_preds))

Test acc (argmax): 0.8541666666666666
Test acc (threshold=0.4): 0.7916666666666666
              precision    recall  f1-score   support

           0       0.86      0.67      0.75        45
           1       0.75      0.90      0.82        51

    accuracy                           0.79        96
   macro avg       0.81      0.78      0.79        96
weighted avg       0.80      0.79      0.79        96

Confusion matrix:
 [[30 15]
 [ 5 46]]


##Extract Embeddings From Model

In [7]:
def extract_embeddings(X_np, batch_size=128):
    model.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32)
    loader = DataLoader(TensorDataset(X_t), batch_size=batch_size, shuffle=False)

    feats = []
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            _, z = model(xb)
            feats.append(z.cpu().numpy())
    return np.concatenate(feats)

F_train = extract_embeddings(X_train)
F_val   = extract_embeddings(X_val)
F_test  = extract_embeddings(X_test)

print(F_train.shape, F_test.shape)  # (N_train, 64), (N_test, 64)

(764, 64) (96, 64)


##Save Features

In [8]:
import os
import numpy as np
import torch

OUT_DIR = os.path.join(os.getcwd(), "artifacts")  # creates ./artifacts next to the notebook
os.makedirs(OUT_DIR, exist_ok=True)

np.save(os.path.join(OUT_DIR, "F_train.npy"), F_train)
np.save(os.path.join(OUT_DIR, "F_val.npy"),   F_val)
np.save(os.path.join(OUT_DIR, "F_test.npy"),  F_test)

np.save(os.path.join(OUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUT_DIR, "y_val.npy"),   y_val)
np.save(os.path.join(OUT_DIR, "y_test.npy"),  y_test)

torch.save(model.state_dict(), os.path.join(OUT_DIR, "cnn_feature_net.pt"))

print("Saved features + model to:", OUT_DIR)

Saved features + model to: c:\Users\arush\NeuronMove_2024-25\notebooks\artifacts
